In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\gbm_glomerulus_summary.csv"
OUTPUT_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\PCA_analysis"

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

PCA_CSV = os.path.join(
    OUTPUT_FOLDER,
    "gbm_glomerulus_PCA_results.csv"
)

PCA_PLOT = os.path.join(
    OUTPUT_FOLDER,
    "PCA_glomerulus_distribution.png"
)

VARIANCE_PLOT = os.path.join(
    OUTPUT_FOLDER,
    "PCA_explained_variance.png"
)

if not os.path.exists(INPUT_CSV):

    raise FileNotFoundError(
        "\nInput file not found:\n"
        + INPUT_CSV
    )

df = pd.read_csv(INPUT_CSV)

print("PCA DIMENSIONALITY REDUCTION OF GBM GLOMERULI")
print("=" * 60)

print(
    f"Glomeruli loaded : {len(df)}"
)

print(
    f"Patients         : "
    f"{df['patient_id'].nunique()}"
)

pca_features = [

    "mean_thickness_nm",
    "median_thickness_nm",
    "std_thickness_nm",
    "min_thickness_nm",
    "max_thickness_nm",
    "number_of_membrane_components"
]

missing = [

    column
    for column in pca_features
    if column not in df.columns

]

if missing:

    raise ValueError(
        "\nThe following PCA variables are missing:\n"
        + "\n".join(missing)
    )


pca_df = df.dropna(
    subset=pca_features
).copy()


print()
print(
    f"Glomeruli used for PCA : "
    f"{len(pca_df)}"
)

X = pca_df[
    pca_features
].to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#PCA
pca = PCA(
    n_components=len(pca_features)
)

X_pca = pca.fit_transform(
    X_scaled
)

explained_variance = (
    pca.explained_variance_ratio_
    * 100
)

cumulative_variance = np.cumsum(
    explained_variance
)

print()
print("PCA EXPLAINED VARIANCE")
print("=" * 60)


for i, variance in enumerate(
    explained_variance,
    start=1
):

    print(
        f"PC{i}: "
        f"{variance:.2f}%"
    )


print()
print(
    f"PC1 + PC2 explained variance : "
    f"{cumulative_variance[1]:.2f}%"
)

for i in range(
    X_pca.shape[1]
):

    pca_df[
        f"PC{i + 1}"
    ] = X_pca[:, i]

pca_df.to_csv(
    PCA_CSV,
    index=False
)

print()
print("PCA results saved:")
print(PCA_CSV)

loadings = pd.DataFrame(

    pca.components_.T,

    index=pca_features,

    columns=[
        f"PC{i + 1}"
        for i in range(
            len(pca_features)
        )
    ]

)

LOADINGS_CSV = os.path.join(
    OUTPUT_FOLDER,
    "PCA_feature_loadings.csv"
)

loadings.to_csv(
    LOADINGS_CSV
)

print()
print("PCA feature loadings saved:")
print(LOADINGS_CSV)


print()
print("PCA FEATURE LOADINGS")
print("=" * 60)

print(
    loadings.round(3).to_string()
)

fig, ax = plt.subplots(
    figsize=(12, 8)
)

patients = sorted(
    pca_df["patient_id"]
    .astype(str)
    .unique()
)

for patient in patients:

    patient_data = pca_df[
        pca_df["patient_id"].astype(str)
        == patient
    ]

    ax.scatter(

        patient_data["PC1"],

        patient_data["PC2"],

        s=65,

        alpha=0.75,

        label=patient
    )

ax.set_xlabel(

    f"PC1 "
    f"({explained_variance[0]:.1f}% variance)",

    fontsize=12
)

ax.set_ylabel(

    f"PC2 "
    f"({explained_variance[1]:.1f}% variance)",

    fontsize=12
)

ax.set_title(

    "PCA Stratification of GBM Glomeruli",

    fontsize=15,

    fontweight="bold",

    pad=15
)

explanation = (

    "How to read this plot\n"

    "• Each point = one glomerulus\n"

    "• Points close together = similar "
    "GBM thickness characteristics\n"

    "• Points far apart = different "
    "GBM characteristics\n"

    "• PC1 and PC2 are the two main "
    "dimensions summarising the data\n"

    "• Point labels/legend indicate "
    "the patient of origin"

)

ax.text(

    1.03,

    0.98,

    explanation,

    transform=ax.transAxes,

    fontsize=9,

    verticalalignment="top",

    bbox=dict(

        boxstyle="round,pad=0.6",

        facecolor="white",

        edgecolor="gray",

        alpha=0.9

    )

)

ax.grid(

    True,

    alpha=0.2,

    linestyle="--"

)


ax.legend(

    title="Patient",

    bbox_to_anchor=(1.03, 0.35),

    loc="upper left",

    fontsize=8

)


plt.subplots_adjust(

    left=0.09,

    right=0.72,

    top=0.90,

    bottom=0.10

)


plt.savefig(

    PCA_PLOT,

    dpi=300,

    bbox_inches="tight",

    pad_inches=0.2

)


plt.close(fig)


print()
print("PCA distribution plot saved:")
print(PCA_PLOT)


fig, ax = plt.subplots(
    figsize=(10, 6)
)


components = np.arange(
    1,
    len(explained_variance) + 1
)

ax.bar(

    components,

    explained_variance,

    alpha=0.7,

    label="Individual variance"

)

ax.plot(

    components,

    cumulative_variance,

    marker="o",

    linewidth=2,

    label="Cumulative variance"
)

ax.set_xlabel(
    "Principal Component",
    fontsize=12
)

ax.set_ylabel(
    "Explained variance (%)",
    fontsize=12
)

ax.set_title(

    "PCA Explained Variance",

    fontsize=14,

    fontweight="bold"
)

ax.set_xticks(
    components
)

ax.grid(

    axis="y",

    alpha=0.2,

    linestyle="--"
)

ax.legend()

plt.tight_layout()
plt.savefig(

    VARIANCE_PLOT,

    dpi=300,

    bbox_inches="tight"
)

plt.close(fig)

print()
print("Explained variance plot saved:")
print(VARIVANCE_PLOT if False else VARIANCE_PLOT)


print()
print("PCA ANALYSIS COMPLETED")
print("=" * 60)

print(
    f"Glomeruli analysed : "
    f"{len(pca_df)}"
)

print(
    f"Patients analysed  : "
    f"{pca_df['patient_id'].nunique()}"
)

print(
    f"PC1 variance       : "
    f"{explained_variance[0]:.2f}%"
)

print(
    f"PC2 variance       : "
    f"{explained_variance[1]:.2f}%"
)

print(
    f"PC1 + PC2          : "
    f"{cumulative_variance[1]:.2f}%"
)

print()
print("Output files:")
print(PCA_CSV)
print(LOADINGS_CSV)
print(PCA_PLOT)
print(VARIANCE_PLOT)

print("=" * 60)

PCA DIMENSIONALITY REDUCTION OF GBM GLOMERULI
Glomeruli loaded : 257
Patients         : 11

Glomeruli used for PCA : 257

PCA EXPLAINED VARIANCE
PC1: 66.26%
PC2: 26.69%
PC3: 7.00%
PC4: 0.04%
PC5: 0.01%
PC6: 0.00%

PC1 + PC2 explained variance : 92.95%

PCA results saved:
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\PCA_analysis\gbm_glomerulus_PCA_results.csv

PCA feature loadings saved:
C:\Users\ishin\OneDrive\Desktop\ish\Glomerulus_analysis\PCA_analysis\PCA_feature_loadings.csv

PCA FEATURE LOADINGS
                                 PC1    PC2    PC3    PC4    PC5    PC6
mean_thickness_nm              0.501  0.012  0.040  0.008 -0.063  0.862
median_thickness_nm            0.501 -0.001  0.048  0.705 -0.378 -0.327
std_thickness_nm               0.041  0.707 -0.677  0.111  0.166  0.009
min_thickness_nm               0.496 -0.094  0.126 -0.029  0.821 -0.233
max_thickness_nm               0.494  0.126 -0.053 -0.700 -0.389 -0.309
number_of_membrane_components -0.073  0.689  0.721 